In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

import plotly.express as px
from sklearn.metrics import confusion_matrix

import plotly.graph_objects as go
from sklearn.metrics import roc_curve
import sklearn.metrics as metrics


c:\Users\veeri\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Here is the table that contains game level metrics about all of the regular season games 
# This is straight from Kaggle 
game_metrics = pd.read_csv('C:/Users/veeri/Documents/Coding_Projects/MM_2026/Data/MRegularSeasonDetailedResults.csv')
game_metrics

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
0,2003,10,1104,68,1328,62,N,0,27,58,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,70,1393,63,N,0,26,62,...,24,9,20,20,25,7,12,8,6,16
2,2003,11,1266,73,1437,61,N,0,24,58,...,26,14,23,31,22,9,12,2,5,23
3,2003,11,1296,56,1457,50,N,0,18,38,...,22,8,15,17,20,9,19,4,3,23
4,2003,11,1400,77,1208,71,N,0,30,61,...,16,17,27,21,15,12,10,7,1,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124524,2026,132,1335,88,1463,84,N,1,29,64,...,24,13,17,17,24,23,9,9,4,16
124525,2026,132,1345,80,1276,72,N,0,30,57,...,24,5,6,11,22,15,7,2,3,18
124526,2026,132,1378,70,1455,55,N,0,23,61,...,19,13,20,12,27,8,14,7,7,19
124527,2026,132,1433,70,1173,62,N,0,23,57,...,21,9,17,8,31,16,11,3,4,18


In [3]:
# To better create matchup rows, I decided to bring in the actual team name so we can put a team 
# to TeamID
teams = pd.read_csv('C:/Users/veeri/Documents/Coding_Projects/MM_2026/Data/MTeamSpellings.csv')
teams

,TeamNameSpelling,TeamID
0,a&m-corpus chris,1394
1,a&m-corpus christi,1394
2,abilene chr,1101
3,abilene christian,1101
4,abilene-christian,1101
...,...,...
1173,youngstown st,1464
1174,youngstown st.,1464
1175,youngstown state,1464
1176,youngstown-st,1464


In [4]:
# Here we want to do a left join so that we can incorporate 
# the winning and losing team names within our pipeline 
game_with_names = game_metrics.merge(
    teams,
    how="left",
    left_on="WTeamID",
    right_on="TeamID"
)

# Also rename the columns so that it makes more intuitive sense 
# when debugging downstream 
game_with_names = game_with_names.rename(columns={
    "TeamNameSpelling": "WTeamName"
})

# Dedup
game_with_names = game_with_names.drop(columns=["TeamID"])


# Same thing as before, except this time with the losing team name 
game_with_names = game_with_names.merge(
    teams,
    how="left",
    left_on="LTeamID",
    right_on="TeamID"
)

# Rename and dedup 
game_with_names = game_with_names.rename(columns={
    "TeamNameSpelling": "LTeamName"
})

game_with_names = game_with_names.drop(columns=["TeamID"])

In [5]:
game_with_names 

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF,WTeamName,LTeamName
0,2003,10,1104,68,1328,62,N,0,27,58,...,22,10,22,8,18,9,2,20,alabama,oklahoma
1,2003,10,1272,70,1393,63,N,0,26,62,...,20,20,25,7,12,8,6,16,memphis,syracuse
2,2003,11,1266,73,1437,61,N,0,24,58,...,23,31,22,9,12,2,5,23,marquette,villanova
3,2003,11,1296,56,1457,50,N,0,18,38,...,15,17,20,9,19,4,3,23,n illinois,winthrop
4,2003,11,1296,56,1457,50,N,0,18,38,...,15,17,20,9,19,4,3,23,northern ill.,winthrop
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1301068,2026,132,1465,63,1430,61,A,0,22,58,...,24,9,24,11,14,6,3,11,cal baptist,utah-valley-state
1301069,2026,132,1465,63,1430,61,A,0,22,58,...,24,9,24,11,14,6,3,11,california baptist,utah valley
1301070,2026,132,1465,63,1430,61,A,0,22,58,...,24,9,24,11,14,6,3,11,california baptist,utah valley st.
1301071,2026,132,1465,63,1430,61,A,0,22,58,...,24,9,24,11,14,6,3,11,california baptist,utah-valley


In [6]:
game_with_names.columns.tolist()

['Season',
 'DayNum',
 'WTeamID',
 'WScore',
 'LTeamID',
 'LScore',
 'WLoc',
 'NumOT',
 'WFGM',
 'WFGA',
 'WFGM3',
 'WFGA3',
 'WFTM',
 'WFTA',
 'WOR',
 'WDR',
 'WAst',
 'WTO',
 'WStl',
 'WBlk',
 'WPF',
 'LFGM',
 'LFGA',
 'LFGM3',
 'LFGA3',
 'LFTM',
 'LFTA',
 'LOR',
 'LDR',
 'LAst',
 'LTO',
 'LStl',
 'LBlk',
 'LPF',
 'WTeamName',
 'LTeamName']

In [7]:

# I am particularly interested in the 2026 season stats. Especially after the wave of NIL, I think that 
# it is important to evaluate a team based on current metrics and not how they did in years past 
mm_data = game_with_names[game_with_names['Season'] == 2026].copy()

# Deduplicate using game-level keys only 
# I noticed that the same game appears 2 or more times becauyse a team can have different spellings
# Here we dedup the mm_data using columns that define a unique game
game_cols = [c for c in mm_data.columns if 'Name' not in c]
mm_data = mm_data.drop_duplicates(subset=game_cols)

print(mm_data.shape)

# In the raw data every row is one game and the columns are split by winner and loser. So for example if Kansas beat 
# Cincinatti, Kansas' stats are under the WFGM and WFGA and Cincy are LFGM and LFGA. But in the next game if Cincy beats a team 
# then they are in the WFGM and WFGA which makes it hard to aggregate data. So we need to reshape the data properly. 

# These are the list of stats that I caare about 
stat_cols = ['FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
             'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']

# This takes every row and renames things from the winner's POV. The winner is the team and the loser is the opponent
# In win_rows we store the stats of the winning team and then also have a binary flag for the winning team 
win_rows = mm_data.rename(columns={
    'WTeamID': 'TeamID', 'WTeamName': 'TeamName', 'WScore': 'Score',
    'LTeamID': 'OppTeamID', 'LTeamName': 'OppTeamName', 'LScore': 'OppScore',
    **{f'W{s}': s for s in stat_cols},
    **{f'L{s}': f'Opp{s}' for s in stat_cols},
    # Add a new column called Win so that we can track if the team won or not 
}).assign(Win=1)

# Same thing as above but here we want to make sure that we track the stats of the losing team 
lose_rows = mm_data.rename(columns={
    'LTeamID': 'TeamID', 'LTeamName': 'TeamName', 'LScore': 'Score',
    'WTeamID': 'OppTeamID', 'WTeamName': 'OppTeamName', 'WScore': 'OppScore',
    **{f'L{s}': s for s in stat_cols},
    **{f'W{s}': f'Opp{s}' for s in stat_cols},
    # Add a field called "Win" where we can track if they won or not 
}).assign(Win=0)




(5647, 36)


In [8]:
win_rows

,Season,DayNum,TeamID,Score,OppTeamID,OppScore,WLoc,NumOT,FGM,FGA,...,OppOR,OppDR,OppAst,OppTO,OppStl,OppBlk,OppPF,TeamName,OppTeamName,Win
1244457,2026,0,1103,85,1241,71,H,0,27,63,...,9,21,14,17,5,9,22,akron,james madison,1
1244459,2026,0,1104,91,1315,62,H,0,31,58,...,4,18,13,10,11,1,19,alabama,north dakota,1
1244461,2026,0,1112,93,1196,87,N,0,30,61,...,15,21,15,14,9,2,23,arizona,florida,1
1244462,2026,0,1116,109,1380,77,H,0,37,73,...,11,21,10,12,3,3,20,arkansas,southern,1
1244469,2026,0,1117,89,1325,85,A,0,27,56,...,6,14,11,10,4,1,19,arkansas st,ohio,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1301033,2026,132,1335,88,1463,84,N,1,29,64,...,17,24,23,9,9,4,16,penn,yale,1
1301035,2026,132,1345,80,1276,72,N,0,30,57,...,11,22,15,7,2,3,18,purdue,michigan,1
1301036,2026,132,1378,70,1455,55,N,0,23,61,...,12,27,8,14,7,7,19,south fla.,wichita st,1
1301060,2026,132,1433,70,1173,62,N,0,23,57,...,8,31,16,11,3,4,18,va commonwealth,dayton,1


In [9]:
lose_rows

,Season,DayNum,OppTeamID,OppScore,TeamID,Score,WLoc,NumOT,OppFGM,OppFGA,...,OR,DR,Ast,TO,Stl,Blk,PF,OppTeamName,TeamName,Win
1244457,2026,0,1103,85,1241,71,H,0,27,63,...,9,21,14,17,5,9,22,akron,james madison,0
1244459,2026,0,1104,91,1315,62,H,0,31,58,...,4,18,13,10,11,1,19,alabama,north dakota,0
1244461,2026,0,1112,93,1196,87,N,0,30,61,...,15,21,15,14,9,2,23,arizona,florida,0
1244462,2026,0,1116,109,1380,77,H,0,37,73,...,11,21,10,12,3,3,20,arkansas,southern,0
1244469,2026,0,1117,89,1325,85,A,0,27,56,...,6,14,11,10,4,1,19,arkansas st,ohio,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1301033,2026,132,1335,88,1463,84,N,1,29,64,...,17,24,23,9,9,4,16,penn,yale,0
1301035,2026,132,1345,80,1276,72,N,0,30,57,...,11,22,15,7,2,3,18,purdue,michigan,0
1301036,2026,132,1378,70,1455,55,N,0,23,61,...,12,27,8,14,7,7,19,south fla.,wichita st,0
1301060,2026,132,1433,70,1173,62,N,0,23,57,...,8,31,16,11,3,4,18,va commonwealth,dayton,0


In [10]:
# Keep only the columns that we need 
keep_cols = (['Season', 'DayNum', 'TeamID', 'TeamName', 'Score',
              'OppTeamID', 'OppTeamName', 'OppScore', 'NumOT', 'Win']
             + stat_cols
             + [f'Opp{s}' for s in stat_cols])

# Concatenate the winning and losing teams information and then also sort my the team id and the day num 
team_games = pd.concat([win_rows[keep_cols], lose_rows[keep_cols]], ignore_index=True)
team_games = team_games.sort_values(['TeamID', 'DayNum']).reset_index(drop=True)
team_games

,Season,DayNum,TeamID,TeamName,Score,OppTeamID,OppTeamName,OppScore,NumOT,Win,...,OppFGA3,OppFTM,OppFTA,OppOR,OppDR,OppAst,OppTO,OppStl,OppBlk,OppPF
0,2026,3,1101,abilene chr,73,1303,ne omaha,71,0,1,...,19,23,31,3,16,10,15,11,3,20
1,2026,11,1101,abilene chr,66,1372,sf austin,76,0,0,...,23,15,26,4,22,14,7,5,5,15
2,2026,15,1101,abilene chr,49,1402,texas st,63,0,0,...,8,16,22,10,23,8,18,11,9,15
3,2026,21,1101,abilene chr,61,1427,texas san antonio,50,0,1,...,15,15,27,3,19,5,14,11,2,15
4,2026,22,1101,abilene chr,58,1456,william & mary,92,0,0,...,16,20,30,5,25,20,14,12,5,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11289,2026,108,1481,new haven,64,1476,stonehill,51,0,1,...,21,5,11,7,25,7,10,2,3,15
11290,2026,110,1481,new haven,84,1192,f dickinson,77,0,1,...,22,11,15,7,18,11,9,5,1,18
11291,2026,113,1481,new haven,67,1384,saint francis (pa),73,0,0,...,16,17,17,3,27,11,11,4,1,17
11292,2026,115,1481,new haven,62,1447,wagner,65,0,0,...,25,12,14,5,29,11,7,4,3,16


In [11]:
# I noticed that there were certain instances where a team would have multiple rows even though they shared the same 
# TeamID: "abilene chr", "abilene christian", "abilene-christian". For this case, we need to choose the first 
# instance and stick with it otherwise our data will be overinflated 
canonical_names = (team_games
                   .groupby('TeamID')['TeamName']
                   .agg(lambda x: x.value_counts().index[0])
                   .reset_index())

# Here we drop any names that we do not use. We also want to make sure that 
# we drop duplicates based on the game_cols subset 
game_cols = [c for c in team_games.columns if 'Name' not in c]
team_games = team_games.drop_duplicates(subset=game_cols)

# Here we join on the canonical names to simply get only the names that are proper for our
# downstream analysis 
team_games = team_games.drop(columns=['TeamName', 'OppTeamName'])
team_games = team_games.merge(canonical_names, on='TeamID', how='left')

# Here we LEFT JOIN team_games with canonical_names to get the opposing team id and opposing team 
# name on the OppTeamID
team_games = team_games.merge(
    canonical_names.rename(columns={'TeamID': 'OppTeamID', 'TeamName': 'OppTeamName'}),
    on='OppTeamID', how='left'
)

# Sort based on the TeamID and DayNum for easy viewing 
team_games = team_games.sort_values(['TeamID', 'DayNum']).reset_index(drop=True)

print(f"team_games shape after dedup: {team_games.shape}")
print(f"Games per team (should be ~30-35):")
print(team_games.groupby('TeamID')['DayNum'].count().describe())

team_games shape after dedup: (11294, 36)
Games per team (should be ~30-35):
count    365.000000
mean      30.942466
std        1.778838
min       25.000000
25%       30.000000
50%       31.000000
75%       32.000000
max       35.000000
Name: DayNum, dtype: float64


In [12]:
team_games

,Season,DayNum,TeamID,Score,OppTeamID,OppScore,NumOT,Win,FGM,FGA,...,OppFTA,OppOR,OppDR,OppAst,OppTO,OppStl,OppBlk,OppPF,TeamName,OppTeamName
0,2026,3,1101,73,1303,71,0,1,25,57,...,31,3,16,10,15,11,3,20,abilene chr,ne omaha
1,2026,11,1101,66,1372,76,0,0,24,57,...,26,4,22,14,7,5,5,15,abilene chr,sf austin
2,2026,15,1101,49,1402,63,0,0,17,55,...,22,10,23,8,18,11,9,15,abilene chr,texas st
3,2026,21,1101,61,1427,50,0,1,20,58,...,27,3,19,5,14,11,2,15,abilene chr,texas san antonio
4,2026,22,1101,58,1456,92,0,0,23,60,...,30,5,25,20,14,12,5,14,abilene chr,william & mary
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11289,2026,108,1481,64,1476,51,0,1,26,61,...,11,7,25,7,10,2,3,15,new haven,stonehill
11290,2026,110,1481,84,1192,77,0,1,30,47,...,15,7,18,11,9,5,1,18,new haven,f dickinson
11291,2026,113,1481,67,1384,73,0,0,24,53,...,17,3,27,11,11,4,1,17,new haven,saint francis (pa)
11292,2026,115,1481,62,1447,65,0,0,19,54,...,14,5,29,11,7,4,3,16,new haven,wagner


In [13]:
# CRITICAL: The key here is that every single game produces 2 rows in team_games 
# One in the perspective of the winning team and another for the losing team 
# We do this so that when we group by the TeamID to compute season averages, we capture all of a team's games 
# win and loss! 

# Here we want to get in game information. Now that we have the team and the opposing team score 
# The great thing is we have the POV from both sides, so it makes aggregating a whole lot easier now 
team_games['PointDiff'] = team_games['Score'] - team_games['OppScore']
team_games['FGPct'] = team_games['FGM'] / team_games['FGA']
team_games['FG3Pct'] = team_games['FGM3'] / team_games['FGA3']
team_games['FTPct'] = team_games['FTM'] / team_games['FTA']
team_games['TotalReb'] = team_games['OR'] + team_games['DR']
team_games['AstToRatio'] = team_games['Ast'] / team_games['TO'].replace(0, 1)

# Opponent shooting (defensive perspective)
team_games['OppFGPct'] = team_games['OppFGM'] / team_games['OppFGA']
team_games['OppFG3Pct'] = team_games['OppFGM3'] / team_games['OppFGA3']
team_games


,Season,DayNum,TeamID,Score,OppTeamID,OppScore,NumOT,Win,FGM,FGA,...,TeamName,OppTeamName,PointDiff,FGPct,FG3Pct,FTPct,TotalReb,AstToRatio,OppFGPct,OppFG3Pct
0,2026,3,1101,73,1303,71,0,1,25,57,...,abilene chr,ne omaha,2,0.438596,0.333333,0.761905,27,1.071429,0.466667,0.315789
1,2026,11,1101,66,1372,76,0,0,24,57,...,abilene chr,sf austin,-10,0.421053,0.200000,0.842105,35,1.200000,0.452830,0.565217
2,2026,15,1101,49,1402,63,0,0,17,55,...,abilene chr,texas st,-14,0.309091,0.333333,0.526316,27,0.428571,0.456522,0.625000
3,2026,21,1101,61,1427,50,0,1,20,58,...,abilene chr,texas san antonio,11,0.344828,0.409091,0.705882,37,0.916667,0.363636,0.200000
4,2026,22,1101,58,1456,92,0,0,23,60,...,abilene chr,william & mary,-34,0.383333,0.222222,0.352941,24,0.789474,0.647059,0.375000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11289,2026,108,1481,64,1476,51,0,1,26,61,...,new haven,stonehill,13,0.426230,0.176471,0.750000,38,1.111111,0.350877,0.285714
11290,2026,110,1481,84,1192,77,0,1,30,47,...,new haven,f dickinson,7,0.638298,0.608696,0.500000,18,1.714286,0.517857,0.363636
11291,2026,113,1481,67,1384,73,0,0,24,53,...,new haven,saint francis (pa),-6,0.452830,0.322581,0.562500,18,2.000000,0.565217,0.250000
11292,2026,115,1481,62,1447,65,0,0,19,54,...,new haven,wagner,-3,0.351852,0.217391,0.904762,29,1.000000,0.400000,0.360000


In [14]:

# Here we want to know if the team has beat other good teams and also has this team lost to bad team. 
# We need to answer this question at every point in the season, without peaking ahead into the future. 
# For example, in game 15, we can only use data from games 1-14

# Here we compute each team's rolling win percentage 
# We groupby() the TeamID and get the wins 
# I used .shift() to make sure we are not using the current game 
team_games['Roll_WinPct_ForSOS'] = (
    team_games.groupby('TeamID')['Win']
    .apply(lambda x: x.shift(1).expanding().mean())
    .reset_index(level=0,drop=True)
)

# For each game, we look up how good each opponent was when they were played 
# Lets say that Duke played UNC. We want to know how good UNC was AT THAT SPECIFIC POINT IN THE SEASON and 
# not the end of the season. The following gives us an idea as to how good the opponent was at that 
# specific point of the season 
opp_strength_lookup = team_games[['TeamID','DayNum','Roll_WinPct_ForSOS']].copy()
opp_strength_lookup = opp_strength_lookup.rename(columns={
    'TeamID':'OppTeamID',
    'Roll_WinPct_ForSOS':'OppStrength'
})


team_games = team_games.merge(opp_strength_lookup,on=['OppTeamID','DayNum'],how='left')

# Here is my version of the NCAA quadrant system. I classified based on 4 different tiers based on win 
# percentage 
# Q1 (70%+ win pct): Elite teams 
# Q2 (55-70% win pct): Good teams 
# Q3 (40-55% win pct): Average teams 
# Q4 (below 40% win pct): Bad teams 
Q1_THRESHOLD = 0.70
Q2_THRESHOLD = 0.55
Q3_THRESHOLD = 0.40

# For every game, we tag each game with the opponents tier 
# This basically tells us if this is a Q1, Q2, Q3, or Q4 opponent 
# By default the team is a Q4 team 
team_games['OppQuad'] = 4 
team_games.loc[team_games['OppStrength']>=Q3_THRESHOLD,'OppQuad'] = 3
team_games.loc[team_games['OppStrength']>=Q2_THRESHOLD,'OppQuad'] = 2
team_games.loc[team_games['OppStrength']>=Q1_THRESHOLD,'OppQuad'] = 1

# Then for each game we create a binary flag that looks at if you won or lost the Q1, Q2, Q3, or Q4 games
# This tells us if the team got quality wins or bad losses or mediocre at both 
team_games['Q1_Win_Flag'] = ((team_games['OppQuad'] == 1)&(team_games['Win'] ==1 )).astype(int)
team_games['Q1_Loss_Flag'] = ((team_games['OppQuad'] == 1)&(team_games['Win'] == 0)).astype(int)
team_games['Q2_Win_Flag'] = ((team_games['OppQuad'] == 2)&(team_games['Win'] == 1)).astype(int)
team_games['Q2_Loss_Flag'] = ((team_games['OppQuad'] ==2 )&(team_games['Win'] == 0)).astype(int)
team_games['Q3_Loss_Flag'] = ((team_games['OppQuad'] == 3)&(team_games['Win'] == 0)).astype(int)
team_games['Q4_Loss_Flag'] = ((team_games['OppQuad'] ==4 )&(team_games['Win'] == 0)).astype(int)

team_games

,Season,DayNum,TeamID,Score,OppTeamID,OppScore,NumOT,Win,FGM,FGA,...,OppFG3Pct,Roll_WinPct_ForSOS,OppStrength,OppQuad,Q1_Win_Flag,Q1_Loss_Flag,Q2_Win_Flag,Q2_Loss_Flag,Q3_Loss_Flag,Q4_Loss_Flag
0,2026,3,1101,73,1303,71,0,1,25,57,...,0.315789,NaN,0.000000,4,0,0,0,0,0,0
1,2026,11,1101,66,1372,76,0,0,24,57,...,0.565217,1.000000,1.000000,1,0,1,0,0,0,0
2,2026,15,1101,49,1402,63,0,0,17,55,...,0.625000,0.500000,0.500000,3,0,0,0,0,1,0
3,2026,21,1101,61,1427,50,0,1,20,58,...,0.200000,0.333333,0.333333,4,0,0,0,0,0,0
4,2026,22,1101,58,1456,92,0,0,23,60,...,0.375000,0.500000,0.600000,2,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11291,2026,108,1481,64,1476,51,0,1,26,61,...,0.285714,0.347826,0.280000,4,0,0,0,0,0,0
11292,2026,110,1481,84,1192,77,0,1,30,47,...,0.363636,0.375000,0.320000,4,0,0,0,0,0,0
11293,2026,113,1481,67,1384,73,0,0,24,53,...,0.250000,0.400000,0.153846,4,0,0,0,0,0,1
11294,2026,115,1481,62,1447,65,0,0,19,54,...,0.360000,0.384615,0.333333,4,0,0,0,0,0,1


In [15]:
# Rename 
rename_map = {
    'Q1_Win_Flag':'Q1_Wins',
    'Q1_Loss_Flag':'Q1_Losses',
    'Q2_Win_Flag':'Q2_Wins',
    'Q2_Loss_Flag':'Q2_Losses',
    'Q3_Loss_Flag':'Q3_Losses',
    'Q4_Loss_Flag':'Q4_Losses',
}

# Same idea as rolling feature except this time we use the sum to get a count of all of the quadrant wins 
# and losses. I also use .shift() to look back a game and not use the current game 
for col,new_name in rename_map.items():
    team_games[new_name] = (
        team_games.groupby('TeamID')[col]
        .apply(lambda x: x.shift(1).expanding().sum())
        .reset_index(level=0,drop=True)
    )

# Get the nuber of Q1Q2 wins (quality wins) and quality losses 
# Get the number of bad losses as well 
team_games['Q1Q2_Wins'] = team_games['Q1_Wins'] + team_games['Q2_Wins']
team_games['Q1Q2_Losses'] = team_games['Q1_Losses'] + team_games['Q2_Losses']
team_games['Q1Q2_WinPct'] = team_games['Q1Q2_Wins'] / (team_games['Q1Q2_Wins'] + team_games['Q1Q2_Losses']).replace(0,1)
team_games['Q3Q4_Losses'] = team_games['Q3_Losses'] + team_games['Q4_Losses']

# This is once again rolling. Here we are getting the strength of schedule for every week. THe OppStrength is 
# the rolling win percentage for the opposing team 
team_games['SOS'] = (
    team_games.groupby('TeamID')['OppStrength']
    .apply(lambda x: x.shift(1).expanding().mean())
    .reset_index(level=0,drop=True)
)

# Clean up temp columns
team_games = team_games.drop(columns=[
    'Roll_WinPct_ForSOS','OppStrength','OppQuad',
    'Q1_Win_Flag','Q1_Loss_Flag','Q2_Win_Flag','Q2_Loss_Flag','Q3_Loss_Flag','Q4_Loss_Flag'
])

# GPT generated Verify
check = team_games.drop_duplicates('TeamID',keep='last')[
    ['TeamID','TeamName','SOS','Q1_Wins','Q1_Losses','Q1Q2_WinPct','Q3Q4_Losses']
].sort_values('SOS',ascending=False)
print("Top 15 teams by SOS (using LAST game — full season profile):")
print(check.head(15).to_string(index=False))
print("\nBottom 10:")
print(check.tail(10).to_string(index=False))

print("\nSanity check vs NET rankings:")
for team in ['duke','houston','gonzaga','michigan','arizona']:
    row = check[check['TeamName']==team]
    if len(row)>0:
        print(f"  {team:<15} SOS:{row['SOS'].values[0]:.3f}  Q1W:{int(row['Q1_Wins'].values[0])}  Q1L:{int(row['Q1_Losses'].values[0])}  Q1Q2%:{row['Q1Q2_WinPct'].values[0]:.3f}  BadL:{int(row['Q3Q4_Losses'].values[0])}")

Top 15 teams by SOS (using LAST game — full season profile):
 TeamID       TeamName      SOS  Q1_Wins  Q1_Losses  Q1Q2_WinPct  Q3Q4_Losses
   1403     texas tech 0.744888     11.0        6.0     0.652174          1.0
   1242         kansas 0.725481     12.0        6.0     0.708333          2.0
   1276       michigan 0.721360     15.0        1.0     0.925926          0.0
   1104        alabama 0.708802      9.0        5.0     0.652174          0.0
   1439  virginia tech 0.693596      8.0        9.0     0.541667          1.0
   1314 north carolina 0.683261     10.0        6.0     0.695652          0.0
   1338           pitt 0.680363      4.0       11.0     0.318182          4.0
   1458      wisconsin 0.678562     11.0        5.0     0.652174          1.0
   1113     arizona st 0.668481      6.0       10.0     0.380952          2.0
   1332         oregon 0.668443      2.0       12.0     0.277778          6.0
   1243      kansas st 0.666408      2.0       12.0     0.217391          1.0
   

In [16]:
# This is a list of every stat we can to compuyte the rolling average for., 
# We can loop over this in our for loop down below 
rolling_stats = ['Score', 'OppScore', 'PointDiff', 'Win',
                 'FGPct', 'FG3Pct', 'FTPct',
                 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
                 'OR', 'DR', 'TotalReb', 'Ast', 'TO', 'Stl', 'Blk', 'PF',
                 'AstToRatio', 'OppFGPct', 'OppFG3Pct']


# Here we are computing rolling season to date averages
# We will loop through each value within the rolling_stats list and compute the average for each 
# Lets sayt that Kansas scored [75, 82, 68, 91, 77] in its first 5 games 
# When we are sitting at game 4 our shifted values will be [NaN, 75, 82, 68] and game 4 true value of 91 will be hidden 
# The .expanding().mean() will compute the growing average after the shift 
# So the rolling score for game 5 will be duke average PPG across first 4 games 
for stat in rolling_stats:
    team_games[f'Roll_{stat}'] = (
        team_games.groupby('TeamID')[stat]
        .apply(lambda x: x.shift(1).expanding().mean())
        .reset_index(level=0, drop=True)
    )

# Same idea as above but here we use the .count(). 
# This simply counts how many games have been played 
team_games['Roll_GamesPlayed'] = (
    team_games.groupby('TeamID')['Win']
    .apply(lambda x: x.shift(1).expanding().count())
    .reset_index(level=0, drop=True)
)

# This is the same as before, except this time we are interested in getting the momenum of the last 10 games 
# We use the window=10 so that we can get the last 5 games. The .mean() once again can be used toe calcaulate the mean 
for stat in rolling_stats:
    team_games[f'L10_{stat}'] = (
        team_games.groupby('TeamID')[stat]
        .apply(lambda x: x.shift(1).rolling(window=10, min_periods=10).mean())
        .reset_index(level=0, drop=True)
    )
    
# Here we want to see what the output looks like 
print(f"team_games shape: {team_games.shape}")
print(f"\nNew columns added: {len([c for c in team_games.columns if c.startswith('Roll_') or c.startswith('L3_')])}")

# Show a sample for one team to verify the rolling logic
sample_team = team_games['TeamID'].iloc[0]
sample = team_games[team_games['TeamID'] == sample_team].head(6)
print(f"\nSample for TeamID {sample_team} (first 6 games):")
print(sample[['DayNum', 'Score', 'Win', 'Roll_Score', 'Roll_Win', 'L10_Score', 'L10_Win']].to_string(index=False))
print("\nNote: Row 1 has NaN rolling stats (no prior games).")



team_games shape: (11296, 104)

New columns added: 25

Sample for TeamID 1101 (first 6 games):
 DayNum  Score  Win  Roll_Score  Roll_Win  L10_Score  L10_Win
      3     73    1         NaN       NaN        NaN      NaN
     11     66    0   73.000000  1.000000        NaN      NaN
     15     49    0   69.500000  0.500000        NaN      NaN
     21     61    1   62.666667  0.333333        NaN      NaN
     22     58    0   62.250000  0.500000        NaN      NaN
     29     71    1   61.400000  0.400000        NaN      NaN

Note: Row 1 has NaN rolling stats (no prior games).


In [17]:
team_games

,Season,DayNum,TeamID,Score,OppTeamID,OppScore,NumOT,Win,FGM,FGA,...,L10_DR,L10_TotalReb,L10_Ast,L10_TO,L10_Stl,L10_Blk,L10_PF,L10_AstToRatio,L10_OppFGPct,L10_OppFG3Pct
0,2026,3,1101,73,1303,71,0,1,25,57,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026,11,1101,66,1372,76,0,0,24,57,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026,15,1101,49,1402,63,0,0,17,55,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026,21,1101,61,1427,50,0,1,20,58,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026,22,1101,58,1456,92,0,0,23,60,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11291,2026,108,1481,64,1476,51,0,1,26,61,...,21.2,26.0,12.5,10.4,5.9,1.5,14.0,1.239592,0.443939,0.346637
11292,2026,110,1481,84,1192,77,0,1,30,47,...,21.3,26.7,11.9,10.2,6.0,2.0,13.4,1.205249,0.448669,0.350209
11293,2026,113,1481,67,1384,73,0,0,24,53,...,21.6,27.0,11.5,10.0,5.9,2.0,13.5,1.198900,0.447728,0.338746
11294,2026,115,1481,62,1447,65,0,0,19,54,...,21.1,26.0,11.3,9.0,6.1,1.9,13.8,1.311400,0.460853,0.335175


In [18]:
team_games.columns.tolist()

['Season',
 'DayNum',
 'TeamID',
 'Score',
 'OppTeamID',
 'OppScore',
 'NumOT',
 'Win',
 'FGM',
 'FGA',
 'FGM3',
 'FGA3',
 'FTM',
 'FTA',
 'OR',
 'DR',
 'Ast',
 'TO',
 'Stl',
 'Blk',
 'PF',
 'OppFGM',
 'OppFGA',
 'OppFGM3',
 'OppFGA3',
 'OppFTM',
 'OppFTA',
 'OppOR',
 'OppDR',
 'OppAst',
 'OppTO',
 'OppStl',
 'OppBlk',
 'OppPF',
 'TeamName',
 'OppTeamName',
 'PointDiff',
 'FGPct',
 'FG3Pct',
 'FTPct',
 'TotalReb',
 'AstToRatio',
 'OppFGPct',
 'OppFG3Pct',
 'Q1_Wins',
 'Q1_Losses',
 'Q2_Wins',
 'Q2_Losses',
 'Q3_Losses',
 'Q4_Losses',
 'Q1Q2_Wins',
 'Q1Q2_Losses',
 'Q1Q2_WinPct',
 'Q3Q4_Losses',
 'SOS',
 'Roll_Score',
 'Roll_OppScore',
 'Roll_PointDiff',
 'Roll_Win',
 'Roll_FGPct',
 'Roll_FG3Pct',
 'Roll_FTPct',
 'Roll_FGM',
 'Roll_FGA',
 'Roll_FGM3',
 'Roll_FGA3',
 'Roll_FTM',
 'Roll_FTA',
 'Roll_OR',
 'Roll_DR',
 'Roll_TotalReb',
 'Roll_Ast',
 'Roll_TO',
 'Roll_Stl',
 'Roll_Blk',
 'Roll_PF',
 'Roll_AstToRatio',
 'Roll_OppFGPct',
 'Roll_OppFG3Pct',
 'Roll_GamesPlayed',
 'L10_Score',
 '

In [19]:
# Get only the rolling features that we know before the game 
feature_cols = ([c for c in team_games.columns if c.startswith('Roll_') or c.startswith('L10_')]
                + ['SOS','Q1_Wins','Q1_Losses','Q2_Wins','Q2_Losses','Q1Q2_Wins','Q1Q2_Losses','Q1Q2_WinPct','Q3Q4_Losses'])

# Here we create a unique key for each game. This will be the identifier or the primary key 
team_games['GameKey'] = team_games.apply(
    lambda r: (r['Season'], r['DayNum'], min(r['TeamID'], r['OppTeamID']), max(r['TeamID'], r['OppTeamID'])),
    axis=1
)

# Here is where we decide who the favorite (Team A) is and the underdog (Team B)
# For each game, we look at both team's rolling win percentage (that we know before the game) 
# and assign the better win record as Team A 
team_games['is_team_a'] = False

for game_key, group in team_games.groupby('GameKey'):
    if len(group) != 2:
        continue
    
    row1 = group.iloc[0]
    row2 = group.iloc[1]
    
    # Compare the win percetnage betweeen the two teams 
    win1 = row1['Roll_Win'] if pd.notna(row1['Roll_Win']) else 0
    win2 = row2['Roll_Win'] if pd.notna(row2['Roll_Win']) else 0
    
    # If the win percentage of team 1 is bigger than team 2, 
    # then team 1 is Team A and team 2 is Team B 
    # vice-versa 
    if win1 > win2:
        team_games.loc[group.index[0], 'is_team_a'] = True
    elif win2 > win1:
        team_games.loc[group.index[1], 'is_team_a'] = True
    else:
        # Tied revcord will just to a random draw 
        if row1['TeamID'] < row2['TeamID']:
            team_games.loc[group.index[0], 'is_team_a'] = True
        else:
            team_games.loc[group.index[1], 'is_team_a'] = True

# Here we seperate the favorite and underdog into two separate dataframes and rename 
# the features columns with an A_ and a B_ for favorite and underdog so we can calculate 
# differential features between underdog and favorite 
team_a = team_games[team_games['is_team_a'] == True].copy()
team_b = team_games[team_games['is_team_a'] == False].copy()

# Rename A the favorite 
a_rename = {c: f'A_{c}' for c in feature_cols}
a_rename.update({'TeamID': 'TeamA_ID', 'TeamName': 'TeamA_Name', 'Win': 'TeamA_Win'})

# Rename B the underdog 
b_rename = {c: f'B_{c}' for c in feature_cols}
b_rename.update({'TeamID': 'TeamB_ID', 'TeamName': 'TeamB_Name'})

team_a = team_a.rename(columns=a_rename)
team_b = team_b.rename(columns=b_rename)

# Merge both sides back into a single matchup row 
matchups = team_a[['GameKey', 'Season', 'DayNum', 'TeamA_ID', 'TeamA_Name', 'TeamA_Win']
                  + [f'A_{c}' for c in feature_cols]].merge(
    team_b[['GameKey', 'TeamB_ID', 'TeamB_Name']
           + [f'B_{c}' for c in feature_cols]],
    on='GameKey',
    how='inner'
)

# alculate delta feature between the two 
for col in feature_cols:
    matchups[f'Diff_{col}'] = matchups[f'A_{col}'] - matchups[f'B_{col}']

# Drop early-season NaN rows
matchups_clean = matchups.dropna(subset=['Diff_Roll_Score', 'Diff_Roll_Win']).copy()

print(f"Total matchups: {len(matchups)}")
print(f"Target distribution:\n{matchups_clean['TeamA_Win'].value_counts()}")
print(f"TeamA win rate: {matchups_clean['TeamA_Win'].mean():.3f}")

Total matchups: 5645
Target distribution:
TeamA_Win
1    3573
0    1851
Name: count, dtype: int64
TeamA win rate: 0.659


In [20]:
matchups_clean

,GameKey,Season,DayNum,TeamA_ID,TeamA_Name,TeamA_Win,A_Roll_Score,A_Roll_OppScore,A_Roll_PointDiff,A_Roll_Win,...,Diff_L10_OppFG3Pct,Diff_SOS,Diff_Q1_Wins,Diff_Q1_Losses,Diff_Q2_Wins,Diff_Q2_Losses,Diff_Q1Q2_Wins,Diff_Q1Q2_Losses,Diff_Q1Q2_WinPct,Diff_Q3Q4_Losses
1,"(2026, 11, 1101, 1372)",2026,11,1101,abilene chr,0,73.000000,71.000000,2.000000,1.000000,...,NaN,-0.500000,-1.0,0.0,0.0,0.0,-1.0,0.0,-1.000000,0.0
2,"(2026, 15, 1101, 1402)",2026,15,1101,abilene chr,0,69.500000,73.500000,-4.000000,0.500000,...,NaN,0.166667,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,-1.0
3,"(2026, 21, 1101, 1427)",2026,21,1101,abilene chr,1,62.666667,70.000000,-7.333333,0.333333,...,NaN,0.333333,0.0,1.0,0.0,0.0,0.0,1.0,0.000000,-1.0
4,"(2026, 29, 1101, 1337)",2026,29,1101,abilene chr,1,61.400000,70.400000,-9.000000,0.400000,...,NaN,-0.163333,0.0,-1.0,-1.0,0.0,-1.0,-1.0,-0.250000,-1.0
5,"(2026, 49, 1101, 1411)",2026,49,1101,abilene chr,1,64.888889,73.888889,-9.000000,0.444444,...,NaN,-0.048629,1.0,1.0,0.0,0.0,1.0,1.0,0.200000,-4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5640,"(2026, 96, 1152, 1481)",2026,96,1481,new haven,0,59.500000,68.350000,-8.850000,0.350000,...,0.003134,-0.081365,0.0,-3.0,0.0,1.0,0.0,-2.0,0.000000,-5.0
5641,"(2026, 108, 1476, 1481)",2026,108,1481,new haven,1,59.913043,67.956522,-8.043478,0.347826,...,0.014146,0.053495,0.0,1.0,1.0,-1.0,1.0,0.0,0.166667,-3.0
5642,"(2026, 110, 1192, 1481)",2026,110,1481,new haven,1,60.083333,67.250000,-7.166667,0.375000,...,0.010044,0.080312,0.0,3.0,1.0,-1.0,1.0,2.0,0.166667,-4.0
5643,"(2026, 113, 1384, 1481)",2026,113,1481,new haven,0,61.040000,67.640000,-6.600000,0.400000,...,-0.019324,0.038198,0.0,2.0,1.0,-3.0,1.0,-1.0,0.166667,-6.0


In [21]:
matchups_clean.columns.tolist()

['GameKey',
 'Season',
 'DayNum',
 'TeamA_ID',
 'TeamA_Name',
 'TeamA_Win',
 'A_Roll_Score',
 'A_Roll_OppScore',
 'A_Roll_PointDiff',
 'A_Roll_Win',
 'A_Roll_FGPct',
 'A_Roll_FG3Pct',
 'A_Roll_FTPct',
 'A_Roll_FGM',
 'A_Roll_FGA',
 'A_Roll_FGM3',
 'A_Roll_FGA3',
 'A_Roll_FTM',
 'A_Roll_FTA',
 'A_Roll_OR',
 'A_Roll_DR',
 'A_Roll_TotalReb',
 'A_Roll_Ast',
 'A_Roll_TO',
 'A_Roll_Stl',
 'A_Roll_Blk',
 'A_Roll_PF',
 'A_Roll_AstToRatio',
 'A_Roll_OppFGPct',
 'A_Roll_OppFG3Pct',
 'A_Roll_GamesPlayed',
 'A_L10_Score',
 'A_L10_OppScore',
 'A_L10_PointDiff',
 'A_L10_Win',
 'A_L10_FGPct',
 'A_L10_FG3Pct',
 'A_L10_FTPct',
 'A_L10_FGM',
 'A_L10_FGA',
 'A_L10_FGM3',
 'A_L10_FGA3',
 'A_L10_FTM',
 'A_L10_FTA',
 'A_L10_OR',
 'A_L10_DR',
 'A_L10_TotalReb',
 'A_L10_Ast',
 'A_L10_TO',
 'A_L10_Stl',
 'A_L10_Blk',
 'A_L10_PF',
 'A_L10_AstToRatio',
 'A_L10_OppFGPct',
 'A_L10_OppFG3Pct',
 'A_SOS',
 'A_Q1_Wins',
 'A_Q1_Losses',
 'A_Q2_Wins',
 'A_Q2_Losses',
 'A_Q1Q2_Wins',
 'A_Q1Q2_Losses',
 'A_Q1Q2_WinPct',
 

In [22]:
print(f"\nNaN counts per column:")
matchups_clean.isna().sum()


NaN counts per column:


GameKey             0
Season              0
DayNum              0
TeamA_ID            0
TeamA_Name          0
                   ..
Diff_Q2_Losses      0
Diff_Q1Q2_Wins      0
Diff_Q1Q2_Losses    0
Diff_Q1Q2_WinPct    0
Diff_Q3Q4_Losses    0
Length: 182, dtype: int64

In [ ]:
# Columns to drop because they do not add value 
drop_cols = ['GameKey', 'Season', 'DayNum', 'TeamA_ID', 'TeamA_Name', 
             'TeamB_ID', 'TeamB_Name']

# Target
target = 'TeamA_Win'

# Final Features I decided to keep 
keep_features = [
    'Diff_Roll_PointDiff',
    'A_Q1Q2_Losses',
    'A_Roll_OppScore',
    'Diff_Roll_Stl',
    'Diff_Roll_OR',
    'A_Roll_PointDiff',
    'B_L10_FGPct',
    'A_L10_Stl',
    'Diff_L10_OppScore',
    'Diff_Roll_OppFGPct',
    'A_Roll_Stl',
    'A_L10_TotalReb',
    'A_Roll_OR',
    'A_L10_Blk',
    'A_Roll_FGM',
    'A_Q1_Wins',
    'Diff_L10_TO',
    'A_Roll_OppFGPct',
    'B_L10_Blk',
    'A_L10_OR',
    'B_Roll_PointDiff',
    'B_L10_FGA3',
    'A_Roll_Blk',
    'A_L10_FGA3',
    'B_L10_FGA',
    'B_L10_FTPct',
    'A_Q2_Losses',
    'Diff_Q3Q4_Losses',
    'B_L10_FGM',
    'Diff_L10_OR',
    'B_Roll_Blk',
]

print(f"Total features: {len(feature_cols)}")
print(f"Total rows: {len(matchups_clean)}")
print(f"Target distribution:\n{matchups_clean[target].value_counts(normalize=True)}")


Total features: 58
Total rows: 5424
Target distribution:
TeamA_Win
1    0.658739
0    0.341261
Name: proportion, dtype: float64


In [24]:
# Sort by DayNum because we want to do a time-based split 
# where we train on old data, validate on middle, and final evaluate on test 
matchups_sorted = matchups_clean.sort_values('DayNum').reset_index(drop=True)

n = len(matchups_sorted)
train_end = int(n * 0.75)
val_end = int(n * 0.90)

# Time based Split 
train_df = matchups_sorted.iloc[:train_end].copy()
val_df = matchups_sorted.iloc[train_end:val_end].copy()
test_df = matchups_sorted.iloc[val_end:].copy()

print(f"Train: {len(train_df)} rows (DayNum {train_df['DayNum'].min()} - {train_df['DayNum'].max()})")
print(f"Val: {len(val_df)} rows (DayNum {val_df['DayNum'].min()} - {val_df['DayNum'].max()})")
print(f"Test: {len(test_df)} rows (DayNum {test_df['DayNum'].min()} - {test_df['DayNum'].max()})")
print(f"\nTarget distribution per split:")
print(f"Train:{train_df[target].mean():.3f}")
print(f"Val: {val_df[target].mean():.3f}")
print(f"Test: {test_df[target].mean():.3f}")

Train: 4068 rows (DayNum 2 - 102)
Val: 813 rows (DayNum 102 - 117)
Test: 543 rows (DayNum 117 - 132)

Target distribution per split:
Train:0.664
Val: 0.631
Test: 0.659


In [25]:
# Make sure to have the features within the keep features list but also tag on the target variable 
train_ag = train_df[keep_features + [target]].copy()
val_ag = val_df[keep_features + [target]].copy()
test_ag = test_df[keep_features + [target]].copy()

train_ag[target] = train_ag[target].astype(int)
val_ag[target] = val_ag[target].astype(int)
test_ag[target] = test_ag[target].astype(int)

print(f"Train features shape: {train_ag.shape}")
print(f"Val features shape:   {val_ag.shape}")
print(f"Test features shape:  {test_ag.shape}")

Train features shape: (4068, 32)
Val features shape:   (813, 32)
Test features shape:  (543, 32)


In [26]:
# use autogluon tabular for predictions 
predictor = TabularPredictor(
    label=target,
    problem_type='binary',
    eval_metric='roc_auc',
    path='mm_autogluon_model_2026_inseason'
).fit(
    train_data=train_ag,
    tuning_data=val_ag,
    time_limit=8000,
    presets='best_quality_v150',
    use_bag_holdout=True,
    num_bag_folds=8,
    num_bag_sets=3,
    num_stack_levels=2,                   
    hyperparameters={
        'XGB': {},
        'GBM': {},
        'CAT': {},
        'RF': {},
        'XT': {}

       
    },
    verbosity=2
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.19045
CPU Count:          16
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       3.65 GB / 15.84 GB (23.0%)
Disk Space Avail:   174.78 GB / 930.80 GB (18.8%)
Presets specified: ['best_quality_v150']
Setting dynamic_stacking from 'auto' to False. Reason: Skip dynamic_stacking when use_bag_holdout is enabled. (use_bag_holdout=True)
Stack configuration (auto_stack=True): num_stack_levels=2, num_bag_folds=8, num_bag_sets=3
Beginning AutoGluon training ... Time limit = 8000s
AutoGluon will save models to "c:\Users\veeri\Documents\Coding_Projects\MM_2026\Notebooks\mm_autogluon_model_2026_inseason"
Train Data Rows:    4068
Train Data Columns: 31
Tuning Data Rows:    813
Tuning Data Columns: 31
Label Column:       TeamA_Win
Problem Type:    

In [27]:
# Get the leaderboard of models 
leaderboard = predictor.leaderboard(test_ag, silent=True)
print(leaderboard.to_string(index=False))

              model  score_test  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
    CatBoost_BAG_L1    0.716548   0.662404     roc_auc        0.058407       0.209843  28.095749                 0.058407                0.209843          28.095749            1       True          3
WeightedEnsemble_L2    0.716548   0.662404     roc_auc        0.060402       0.209843  28.139232                 0.001995                0.000000           0.043483            2       True          6
WeightedEnsemble_L3    0.707247   0.659597     roc_auc        0.662500       3.190005  84.510362                 0.014811                0.000000           0.035744            3       True         12
    CatBoost_BAG_L2    0.706930   0.659461     roc_auc        0.498769       2.158760  75.924089                 0.087217                0.207683          29.576723            2       True          9


In [28]:
# Get feature importance 
importance = predictor.feature_importance(test_ag)
print("Top 20 Most Important Features:")
print(importance.head(25))

Computing feature importance via permutation shuffling for 31 features using 543 rows with 5 shuffle sets...
	148.74s	= Expected runtime (29.75s per shuffle set)
	13.53s	= Actual runtime (Completed 5 of 5 shuffle sets)


Top 20 Most Important Features:
                     importance    stddev   p_value  n  p99_high   p99_low
Diff_Roll_PointDiff    0.084666  0.025191  0.000839  5  0.136533  0.032798
B_L10_FGA3             0.008205  0.001594  0.000163  5  0.011487  0.004922
B_L10_FGPct            0.005209  0.001136  0.000255  5  0.007548  0.002871
Diff_Roll_Stl          0.004025  0.002152  0.006949  5  0.008457 -0.000406
Diff_Roll_OR           0.002953  0.005161  0.134936  5  0.013580 -0.007673
Diff_Q3Q4_Losses       0.002730  0.001256  0.004140  5  0.005316  0.000144
A_Q2_Losses            0.002340  0.001625  0.016145  5  0.005687 -0.001006
A_Q1Q2_Losses          0.002023  0.002051  0.046033  5  0.006247 -0.002200
A_Roll_Blk             0.001984  0.001424  0.017838  5  0.004916 -0.000948
A_Roll_OppScore        0.001845  0.000717  0.002264  5  0.003322  0.000368
B_L10_FGA              0.001839  0.001374  0.020102  5  0.004668 -0.000990
B_Roll_Blk             0.001739  0.001091  0.011755  5  0.003987 -0.

In [29]:
# Save
importance.to_csv('feature_importance.csv', index=True)

In [30]:
# Make sure we are using the right model 
print(predictor.model_best)


WeightedEnsemble_L4


In [31]:
# Here we use a threshold of 0.5 as our basis 
THRESHOLD = 0.58

# Get the true values for the test set 
y_true = test_ag[target]
# Get the prediction by the model on the test set 
y_proba = predictor.predict_proba(test_ag.drop(columns=[target]))[1]
# If the predicted probability is greater than the threshold then we predict favorite wins (1)
# otherwise we predict underdog wins (0)
y_pred = (y_proba >= THRESHOLD).astype(int)

# Get other metrics 
auc = roc_auc_score(y_true, y_proba)
print(f"Threshold: {THRESHOLD}")
print(f"ROC AUC: {auc:.4f}\n")
print("Classification Report (Test Set):")
print(classification_report(y_true, y_pred, target_names=['Underdog Win', 'Favorite Win']))


# Build out the confusion matrix
labels = ['Underdog Win', 'Favorite Win']
cm = confusion_matrix(y_true, y_pred)
fig = px.imshow(
    cm,
    text_auto=True,               
    x=labels,                      
    y=labels,                    
    labels=dict(x="Predicted", y="Actual", color="Count"),
    color_continuous_scale='Blues',
    title=f'March Madness Model — Test Set (Threshold = {THRESHOLD})<br>Confusion Matrix'
)

fig.update_layout(width=600, height=500)
fig.show()



Threshold: 0.58
ROC AUC: 0.7013

Classification Report (Test Set):
              precision    recall  f1-score   support

Underdog Win       0.52      0.62      0.56       185
Favorite Win       0.78      0.70      0.74       358

    accuracy                           0.67       543
   macro avg       0.65      0.66      0.65       543
weighted avg       0.69      0.67      0.68       543



In [32]:

# Here we get the ROC metrics 
fpr, tpr, thresholds = roc_curve(y_true, y_proba)
score_auc = metrics.auc(fpr, tpr)
idx = (thresholds >= THRESHOLD).sum() - 1

# Create the figure 
fig = go.Figure()

# Add the main ROC Curve
fig.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode='lines',
    name=f'AUC = {score_auc:.4f}',
    line=dict(color='#1E88E5', width=3),
    fill='tozeroy',
    fillcolor='rgba(30,136,229,0.1)',
    hovertemplate='FPR: %{x:.3f}<br>TPR: %{y:.3f}<extra></extra>'
))

# Add the Random Baseline 
fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode='lines',
    name='Random (0.5)',
    line=dict(color='#9E9E9E', dash='dash', width=1.5)
))

# Add the specific Threshold Point
fig.add_trace(go.Scatter(
    x=[fpr[idx]], y=[tpr[idx]],
    mode='markers+text',
    name=f'Threshold = {THRESHOLD}',
    marker=dict(color='#E53935', size=14, symbol='circle',
                line=dict(color='white', width=2)),
    text=[f'  TPR={tpr[idx]:.2f}, FPR={fpr[idx]:.2f}'],
    textposition='middle right',
    textfont=dict(size=11, color='#E53935')
))

# Update layout and styling
fig.update_layout(
    title=dict(
        text=f'March Madness Model — ROC Curve<br><span style="font-size:13px;color:#666">Threshold = {THRESHOLD} | AUC = {score_auc:.4f}</span>',
        x=0.5,
        xanchor='center',
        font=dict(size=18)
    ),
    xaxis=dict(
        title='False Positive Rate',
        range=[0, 1],
        constrain='domain',
        gridcolor='#E0E0E0',
        zeroline=False
    ),
    yaxis=dict(
        title='True Positive Rate',
        range=[0, 1.05],
        scaleanchor='x',
        scaleratio=1,
        gridcolor='#E0E0E0',
        zeroline=False
    ),
    width=700,
    height=650,
    legend=dict(
        yanchor='bottom', y=0.02,
        xanchor='right', x=0.98,
        bgcolor='rgba(255,255,255,0.9)',
        bordercolor='#E0E0E0',
        borderwidth=1,
        font=dict(size=12)
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    margin=dict(t=80, b=60, l=60, r=40)
)

fig.show()

In [33]:
importance

,importance,stddev,p_value,n,p99_high,p99_low
Diff_Roll_PointDiff,0.084666,0.025191,0.000839,5,0.136533,0.032798
B_L10_FGA3,0.008205,0.001594,0.000163,5,0.011487,0.004922
B_L10_FGPct,0.005209,0.001136,0.000255,5,0.007548,0.002871
Diff_Roll_Stl,0.004025,0.002152,0.006949,5,0.008457,-0.000406
Diff_Roll_OR,0.002953,0.005161,0.134936,5,0.013580,-0.007673
Diff_Q3Q4_Losses,0.002730,0.001256,0.004140,5,0.005316,0.000144
A_Q2_Losses,0.002340,0.001625,0.016145,5,0.005687,-0.001006
A_Q1Q2_Losses,0.002023,0.002051,0.046033,5,0.006247,-0.002200
A_Roll_Blk,0.001984,0.001424,0.017838,5,0.004916,-0.000948
A_Roll_OppScore,0.001845,0.000717,0.002264,5,0.003322,0.000368


In [34]:
# See some samples 
test_results = test_df[['DayNum', 'TeamA_Name', 'TeamB_Name', 'TeamA_Win']].copy()
test_results['Pred_TeamA_Win'] = y_pred.values
test_results['Prob_TeamA_Win'] = y_proba.values
test_results['Correct'] = (test_results['TeamA_Win'] == test_results['Pred_TeamA_Win']).astype(int)

print(f"Overall accuracy: {test_results['Correct'].mean():.3f}")
print(f"\nSample predictions:")
print(test_results.head(20).to_string(index=False))

Overall accuracy: 0.674

Sample predictions:
 DayNum        TeamA_Name        TeamB_Name  TeamA_Win  Pred_TeamA_Win  Prob_TeamA_Win  Correct
    117           buffalo        c michigan          0               1        0.674321        0
    117        new mexico      san diego st          1               1        0.641054        1
    117          nicholls      se louisiana          1               0        0.520663        0
    117     bowling green     massachusetts          1               0        0.529448        0
    117            wagner        chicago st          1               0        0.547666        0
    117 tex.-pan american    east texas a&m          1               1        0.700376        1
    117               liu       f dickinson          1               1        0.687700        1
    117       saint louis          duquesne          1               1        0.886052        1
    117            nevada  nevada-las vegas          0               1        0.673424     

In [35]:
# Try to find the best threshold for us to win us those 50/50 games in Round 1 
# This threshold will also be used to genreate the confusion matrices when re-training 
for threshold in [0.5, 0.6, 0.7, 0.8]:
    confident = test_results[
        (test_results['Prob_TeamA_Win'] >= threshold) | 
        (test_results['Prob_TeamA_Win'] <= (1 - threshold))
    ]
    if len(confident) > 0:
        acc = confident['Correct'].mean()
        print(f"Threshold {threshold:.1f}: {len(confident)} games, accuracy {acc:.3f}")

Threshold 0.5: 543 games, accuracy 0.674
Threshold 0.6: 306 games, accuracy 0.784
Threshold 0.7: 180 games, accuracy 0.822
Threshold 0.8: 64 games, accuracy 0.875


In [36]:
# Save team_games for use in the inference notebook
team_games.to_csv('C:/Users/veeri/Documents/Coding_Projects/MM_2026/Notebooks/team_games_processed.csv', index=False)
print(f"Saved team_games: {team_games.shape}")

Saved team_games: (11296, 106)
